<a href="https://colab.research.google.com/github/othmane-belghazi/NLP/blob/main/catboost_resiliation_tarifaire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modèle CatBoost — Probabilité de résiliation tarifaire

Pipeline complet :

1. Configuration des variables (continues / binaires / catégorielles)
2. Préparation minimale (les NaN des **continues** sont laissés tels quels → gérés nativement par CatBoost ; les catégorielles manquantes sont déjà en `"MISSING"`)
3. **Contrainte de monotonie** sur les variables tarifaires (↑ tarif ⇒ ↑ proba de résiliation)
4. **Pondération de la classe rare** (`scale_pos_weight`)
5. **Validation croisée stratifiée + optimisation des hyperparamètres** (Optuna, métrique PR-AUC adaptée au déséquilibre)
6. Entraînement final avec early stopping
7. **Visualisations de validation** (ROC, PR, calibration, KS, lift, importance, SHAP)
8. **Analyse d'élasticité** : réponse de la proba au tarif + élasticité prix
9. **Vérification de la monotonie** apprise

> ⚠️ Dépendances : `pip install catboost optuna shap scikit-learn matplotlib seaborn`
> Utilisez une version récente de CatBoost (≥ 1.0) : les contraintes de monotonie y cohabitent avec les variables catégorielles.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, brier_score_loss, f1_score,
)
from sklearn.calibration import calibration_curve
import optuna

import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

## 2. Configuration — À ADAPTER À VOS COLONNES

Renseignez ici vos listes de features. C'est le **seul endroit** à modifier pour brancher vos données.


In [ ]:
TARGET = "resilie"   # colonne cible binaire (1 = résilié, 0 = actif)

# --- Renseignez vos colonnes -----------------------------------------
continuous_features  = [
    # ex : "anciennete", "conso_moyenne", "nb_contacts_sav", ...
]
binary_features      = [
    # ex : "a_option_X", "client_pro", ...  (valeurs 0/1)
]
categorical_features = [
    # ex : "segment", "region", "canal_souscription", ...
    # (les manquants y sont déjà encodés en "MISSING")
]

# Variables TARIFAIRES soumises à la monotonie CROISSANTE :
# plus la variable augmente, plus la proba de résiliation augmente  -> +1
tariff_features = [
    # ex : "tarif_mensuel", "evolution_tarif_pct", "ecart_tarif_marche"
]

# Variable tarifaire principale utilisée pour l'analyse d'élasticité
PRICE_FEATURE = tariff_features[0] if tariff_features else None

## 3. Chargement des données

Remplacez ce bloc par le chargement de votre jeu de données déjà préparé.
Un générateur synthétique optionnel est fourni pour exécuter le notebook de bout en bout immédiatement.

In [ ]:
USE_SYNTHETIC = True   # passez à False quand vous branchez vos vraies données

if USE_SYNTHETIC:
    # --- jeu de démonstration (à supprimer en production) ---
    n = 40_000
    rng = np.random.default_rng(RANDOM_STATE)
    tarif = rng.gamma(4, 8, n)                     # variable tarifaire
    anciennete = rng.exponential(30, n)
    conso = rng.normal(100, 30, n)
    seg = rng.choice(["A", "B", "C", "MISSING"], n, p=[.4, .3, .25, .05])
    region = rng.choice(["Nord", "Sud", "Est", "Ouest"], n)
    opt = rng.integers(0, 2, n)
    # proba de résiliation croissante avec le tarif (vérité terrain monotone)
    logit = -5.0 + 0.06 * tarif - 0.02 * anciennete + 0.3 * opt
    p = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, p)
    df = pd.DataFrame({
        "tarif_mensuel": tarif, "anciennete": anciennete, "conso_moyenne": conso,
        "segment": seg, "region": region, "a_option_X": opt, TARGET: y,
    })
    # on injecte des NaN dans une continue pour tester la gestion native
    df.loc[rng.random(n) < 0.1, "conso_moyenne"] = np.nan

    continuous_features  = ["tarif_mensuel", "anciennete", "conso_moyenne"]
    binary_features      = ["a_option_X"]
    categorical_features = ["segment", "region"]
    tariff_features      = ["tarif_mensuel"]
    PRICE_FEATURE        = "tarif_mensuel"
else:
    df = pd.read_parquet("votre_dataset.parquet")   # <-- votre fichier

print(df.shape)
df.head()

## 4. Préparation minimale des types

- **Catégorielles** → `str`, manquants en `"MISSING"` (CatBoost n'accepte pas de NaN sur les catégorielles).
- **Continues** → numériques, les **NaN sont conservés** (gestion native via `nan_mode`).
- **Binaires** → numériques 0/1.

In [ ]:
for c in categorical_features:
    df[c] = df[c].astype("object").where(df[c].notna(), "MISSING").astype(str)

for c in continuous_features:
    df[c] = pd.to_numeric(df[c], errors="coerce")   # NaN laissés tels quels

for c in binary_features:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Taux de résiliation : {:.3%}".format(df[TARGET].mean()))
print("NaN par feature continue :")
print(df[continuous_features].isna().mean().round(3))

## 5. Split stratifié & contraintes

In [ ]:
features = continuous_features + binary_features + categorical_features
X = df[features].copy()
y = df[TARGET].astype(int).values

cat_idx = [features.index(c) for c in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Contrainte de monotonie croissante (+1) sur les variables tarifaires
monotone_constraints = {f: 1 for f in tariff_features}

# Pondération de la classe rare
neg, pos = np.bincount(y_train)
base_spw = neg / pos
print(f"scale_pos_weight de base (neg/pos) = {base_spw:.1f}")
print("Contraintes de monotonie :", monotone_constraints)

## 6. Validation croisée + optimisation des hyperparamètres (Optuna)

- StratifiedKFold (préserve la rareté de la classe dans chaque fold)
- Métrique optimisée : **PR-AUC** (`average_precision`), plus pertinente que l'AUC quand le positif est très rare
- Contraintes de monotonie, pondération de classe et `nan_mode` appliqués à chaque fold
- `boosting_type="Plain"` requis avec les contraintes de monotonie

In [ ]:
N_TRIALS = 40   # augmentez pour une recherche plus fine
N_SPLITS = 5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 2000,
        "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth":              trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg":        trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength":    trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature":trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count":       trial.suggest_int("border_count", 32, 254),
        "scale_pos_weight":   trial.suggest_float("scale_pos_weight", base_spw*0.5, base_spw*3.0),
        "monotone_constraints": monotone_constraints,
        "boosting_type": "Plain",
        "nan_mode": "Min",            # gestion native des NaN des continues
        "random_seed": RANDOM_STATE,
        "allow_writing_files": False,
        "verbose": False,
    }
    scores = []
    for tr, va in skf.split(X_train, y_train):
        Xtr, Xva = X_train.iloc[tr], X_train.iloc[va]
        ytr, yva = y_train[tr], y_train[va]
        train_pool = Pool(Xtr, ytr, cat_features=cat_idx)
        val_pool   = Pool(Xva, yva, cat_features=cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(train_pool, eval_set=val_pool,
              early_stopping_rounds=100, use_best_model=True, verbose=False)
        p = m.predict_proba(Xva)[:, 1]
        scores.append(average_precision_score(yva, p))   # PR-AUC
    return float(np.mean(scores))

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nMeilleur PR-AUC (CV) : {:.4f}".format(study.best_value))
best_params = study.best_params
best_params

## 7. Modèle final (early stopping sur une validation interne)

In [ ]:
final_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 4000,
    "monotone_constraints": monotone_constraints,
    "boosting_type": "Plain",
    "nan_mode": "Min",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": 200,
    **best_params,
}

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
val_pool   = Pool(X_val, y_val, cat_features=cat_idx)

model = CatBoostClassifier(**final_params)
model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=150, use_best_model=True)

print("Itérations retenues :", model.get_best_iteration())

In [ ]:
# Prédictions sur le test
test_pool = Pool(X_test, y_test, cat_features=cat_idx)
proba_test = model.predict_proba(X_test)[:, 1]

auc  = roc_auc_score(y_test, proba_test)
ap   = average_precision_score(y_test, proba_test)
brier = brier_score_loss(y_test, proba_test)
print(f"Test AUC      : {auc:.4f}")
print(f"Test PR-AUC   : {ap:.4f}")
print(f"Brier score   : {brier:.4f}  (plus bas = mieux calibré)")

## 8. Visualisations de validation

### 8.1 Courbes ROC & Précision-Rappel

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_test)
prec, rec, _ = precision_recall_curve(y_test, proba_test)
baseline = y_test.mean()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
ax[0].plot([0, 1], [0, 1], "--", c="grey")
ax[0].set(xlabel="Taux faux positifs", ylabel="Taux vrais positifs", title="Courbe ROC")
ax[0].legend()

ax[1].plot(rec, prec, label=f"PR-AUC = {ap:.3f}", color="darkorange")
ax[1].axhline(baseline, ls="--", c="grey", label=f"Hasard = {baseline:.3f}")
ax[1].set(xlabel="Rappel", ylabel="Précision", title="Courbe Précision-Rappel")
ax[1].legend()
plt.tight_layout(); plt.show()

### 8.2 Calibration & distribution des probabilités

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(mean_pred, frac_pos, "o-", label="Modèle")
ax[0].plot([0, 1], [0, 1], "--", c="grey", label="Parfaitement calibré")
ax[0].set(xlabel="Proba prédite moyenne", ylabel="Fréquence observée",
          title=f"Courbe de calibration (Brier={brier:.3f})")
ax[0].legend()

ax[1].hist(proba_test[y_test == 0], bins=50, alpha=.6, density=True, label="Actifs (0)")
ax[1].hist(proba_test[y_test == 1], bins=50, alpha=.6, density=True, label="Résiliés (1)")
ax[1].set(xlabel="Proba prédite", ylabel="Densité",
          title="Séparation des distributions")
ax[1].legend()
plt.tight_layout(); plt.show()

### 8.3 Analyse du seuil & matrice de confusion

In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
f1s = [f1_score(y_test, (proba_test >= t).astype(int)) for t in thresholds]
best_t = thresholds[int(np.argmax(f1s))]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
precs = [ (proba_test>=t).astype(int) for t in thresholds ]
ax[0].plot(thresholds, f1s, label="F1")
ax[0].axvline(best_t, ls="--", c="red", label=f"Seuil F1 optimal = {best_t:.2f}")
ax[0].set(xlabel="Seuil", ylabel="F1", title="F1 vs seuil de décision")
ax[0].legend()

y_pred = (proba_test >= best_t).astype(int)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[1],
            xticklabels=["Actif", "Résilié"], yticklabels=["Actif", "Résilié"])
ax[1].set(xlabel="Prédit", ylabel="Réel", title=f"Matrice de confusion @ {best_t:.2f}")
plt.tight_layout(); plt.show()

print(classification_report(y_test, y_pred, target_names=["Actif", "Résilié"]))

### 8.4 Statistique KS & courbe de lift / gains cumulés

In [ ]:
# KS
order = np.argsort(proba_test)
cum_pos = np.cumsum(y_test[order]) / y_test.sum()
cum_neg = np.cumsum(1 - y_test[order]) / (len(y_test) - y_test.sum())
ks = np.max(np.abs(cum_pos - cum_neg))

# Gains cumulés (tri décroissant des scores)
desc = np.argsort(-proba_test)
gains = np.cumsum(y_test[desc]) / y_test.sum()
pct_pop = np.arange(1, len(y_test) + 1) / len(y_test)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(cum_neg, label="Cumul actifs")
ax[0].plot(cum_pos, label="Cumul résiliés")
ax[0].set(title=f"Courbe KS = {ks:.3f}", xlabel="Échantillons triés", ylabel="Proportion cumulée")
ax[0].legend()

ax[1].plot(pct_pop, gains, label="Modèle")
ax[1].plot([0, 1], [0, 1], "--", c="grey", label="Hasard")
ax[1].set(title="Gains cumulés", xlabel="% population ciblée", ylabel="% résiliés captés")
ax[1].legend()
plt.tight_layout(); plt.show()
print(f"KS = {ks:.3f}")

### 8.5 Importance des variables

In [ ]:
imp = (pd.Series(model.get_feature_importance(test_pool), index=features)
       .sort_values())

plt.figure(figsize=(8, max(4, 0.4*len(features))))
imp.plot(kind="barh", color="steelblue")
plt.title("Importance des variables (PredictionValuesChange)")
plt.xlabel("Importance")
plt.tight_layout(); plt.show()

### 8.6 Valeurs SHAP

Calcul natif via CatBoost (robuste avec les variables catégorielles).

In [ ]:
shap_raw = model.get_feature_importance(test_pool, type="ShapValues")
shap_values = shap_raw[:, :-1]      # dernière colonne = valeur de base

mean_abs = (pd.Series(np.abs(shap_values).mean(0), index=features)
            .sort_values())

plt.figure(figsize=(8, max(4, 0.4*len(features))))
mean_abs.plot(kind="barh", color="indianred")
plt.title("Impact moyen sur la sortie (|SHAP| moyen)")
plt.xlabel("|valeur SHAP| moyenne")
plt.tight_layout(); plt.show()

# Beeswarm optionnel (peut échouer sur features catégorielles selon version de shap)
try:
    import shap
    shap.summary_plot(shap_values, X_test, feature_names=features, show=True)
except Exception as e:
    print("Beeswarm SHAP ignoré :", e)

## 9. Analyse d'élasticité

On fait varier le **tarif** de ±20 % sur l'échantillon test (toutes choses égales par ailleurs)
et on mesure la réponse de la probabilité moyenne de résiliation, puis on en déduit
l'**élasticité prix** = (%Δ proba) / (%Δ tarif).

In [ ]:
assert PRICE_FEATURE is not None, "Définissez PRICE_FEATURE / tariff_features."

deltas = np.linspace(-0.20, 0.20, 9)   # variations relatives du tarif
mean_proba = []
for d in deltas:
    tmp = X_test.copy()
    tmp[PRICE_FEATURE] = tmp[PRICE_FEATURE] * (1 + d)
    mean_proba.append(model.predict_proba(tmp[features])[:, 1].mean())
mean_proba = np.array(mean_proba)

# Élasticité point central (différence centrée autour de 0 %)
p0 = mean_proba[len(deltas)//2]
step = deltas[1] - deltas[0]
dP = (mean_proba[len(deltas)//2 + 1] - mean_proba[len(deltas)//2 - 1]) / (2*step)
elasticity = (dP / p0)   # car d(prix)/prix = step en relatif -> déjà normalisé

plt.figure(figsize=(7, 5))
plt.plot(deltas*100, mean_proba, "o-")
plt.axvline(0, ls="--", c="grey")
plt.title(f"Réponse de la proba de résiliation au tarif\nÉlasticité ≈ {elasticity:.2f}")
plt.xlabel(f"Variation du tarif (%) — {PRICE_FEATURE}")
plt.ylabel("Proba moyenne de résiliation")
plt.tight_layout(); plt.show()

print(f"Élasticité prix de la résiliation ≈ {elasticity:.3f}")
print("(une hausse de 1% du tarif fait varier la proba de résiliation de ~{:.2f}%)".format(elasticity))

### 9.1 Dépendance SHAP sur le tarif

In [ ]:
pf = features.index(PRICE_FEATURE)
plt.figure(figsize=(7, 5))
plt.scatter(X_test[PRICE_FEATURE], shap_values[:, pf], s=6, alpha=.3)
plt.axhline(0, ls="--", c="grey")
plt.title(f"Dépendance SHAP — {PRICE_FEATURE}")
plt.xlabel(PRICE_FEATURE); plt.ylabel("Contribution SHAP (vers résiliation)")
plt.tight_layout(); plt.show()

## 10. Vérification de la monotonie apprise

On découpe le tarif en déciles et on trace la proba moyenne prédite par bin.
La courbe doit être **non décroissante** (contrainte respectée).

In [ ]:
bins = pd.qcut(X_test[PRICE_FEATURE], 10, duplicates="drop")
prob_by_bin = pd.Series(proba_test, index=X_test.index).groupby(bins, observed=True).mean()

is_monotone = np.all(np.diff(prob_by_bin.values) >= -1e-9)

plt.figure(figsize=(8, 5))
plt.plot(range(len(prob_by_bin)), prob_by_bin.values, "o-")
plt.title(f"Proba de résiliation par décile de tarif — monotone : {is_monotone}")
plt.xlabel(f"Décile de {PRICE_FEATURE} (croissant)")
plt.ylabel("Proba moyenne prédite")
plt.tight_layout(); plt.show()

print("Monotonie croissante respectée :", bool(is_monotone))

## 11. Sauvegarde du modèle

In [ ]:
model.save_model("catboost_resiliation.cbm")
print("Modèle sauvegardé -> catboost_resiliation.cbm")

# Rechargement :
# from catboost import CatBoostClassifier
# m = CatBoostClassifier(); m.load_model("catboost_resiliation.cbm")